# Project 02 – StyleGAN2 Face Generation
Progressive training: **256 → 512 → 1024**

**Setup flow**
1. `git clone` → code (always up-to-date with repo)
2. Google Drive mount → **checkpoint backup** only
3. Colab 파일 브라우저로 `/content/` 에 zip 업로드 → 압축 해제
4. WandB login → train

## 1. Install packages + clone repo

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/jyun-chae/skku-2-openai_pa2.git'
REPO_BRANCH = 'main'
# ────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

# ── Packages ─────────────────────────────────────────────────────────────────
!pip install -q wandb pytorch-fid pyyaml scikit-image onnx
!pip install -q 'torch>=2.2.0' torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade

# ── Clone or pull ─────────────────────────────────────────────────────────────
REPO_DIR = '/content/project02'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', REPO_BRANCH], check=True)
    print('Pulled latest.')
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
        check=True,
    )
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print(f'torch={torch.__version__}  CUDA={torch.cuda.is_available()}',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Mount Google Drive (checkpoint backup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── USER CONFIG ──────────────────────────────────────────────────────────────
DRIVE_DIR      = '/content/drive/MyDrive/project02'
DRIVE_DATA_DIR = f'{DRIVE_DIR}/data'           # chunk files live here
DRIVE_CKPT_DIR = f'{DRIVE_DIR}/checkpoints'    # checkpoint backup
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Drive ready.')

## 3. Data setup

Drive의 `project02/data/` 에 chunk 파일을 업로드한 뒤 **섹션별로** 실행합니다.

**업로드해야 할 파일 (해상도별)**

| 해상도 | train chunks | valid chunks |
|---|---|---|
| 256 | `train_50k_256.zip.000~001` (2개) | `valid_10k_256.zip` (단일) |
| 512 | `train_50k_512.zip.000~005` (6개) | `valid_10k_512.zip.000~001` (2개) |
| 1024 | `train_50k_1024.zip.000~019` (20개) | `valid_10k_1024.zip.000~003` (4개) |

`split_manifest.json` 도 반드시 업로드하세요.

In [ ]:
### 3-a. Config
# ── USER CONFIG ──────────────────────────────────────────────────────────────
DATA_RES = 256   # 256 / 512 / 1024
# ────────────────────────────────────────────────────────────────────────────

import json, shutil
from pathlib import Path
from parallel_unzip import parallel_unzip

EXTRACT_TO = f'/content/ffhq_{DATA_RES}'
train_dir  = Path(EXTRACT_TO) / 'train'
valid_dir  = Path(EXTRACT_TO) / 'valid'
train_dir.mkdir(parents=True, exist_ok=True)
valid_dir.mkdir(parents=True, exist_ok=True)

# Manifest for size verification
_manifest_src = Path(DRIVE_DATA_DIR) / 'split_manifest.json'
_manifest = json.loads(_manifest_src.read_text()) if _manifest_src.exists() else {}
print(f'DATA_RES={DATA_RES}  DRIVE_DATA_DIR={DRIVE_DATA_DIR}')

### 3-b. Drive → /content/ 복사 (chunk 1개씩, 반복 실행)

Drive에 chunk 1개 업로드 → `CHUNK_NAME` 바꾸고 셀 실행 → 반복

In [ ]:
### 3-b. Drive → /content/ 복사  ← chunk 1개 업로드할 때마다 이름 바꿔서 실행

# ── USER CONFIG ──────────────────────────────────────────────────────────────
CHUNK_NAME = 'train_50k_256.zip.000'   # ← 매번 여기만 바꾸세요
# ────────────────────────────────────────────────────────────────────────────

src = Path(DRIVE_DATA_DIR) / CHUNK_NAME
dst = Path('/content') / CHUNK_NAME

assert src.exists(), (
    f'\n{src} 가 Drive에 없습니다.\n'
    f'Drive의 project02/data/ 폴더에 {CHUNK_NAME} 을 먼저 업로드하세요.'
)

# 이미 복사된 파일이면 건너뜀
if dst.exists() and dst.stat().st_size == src.stat().st_size:
    print(f'이미 존재: {dst}  ({dst.stat().st_size/1e9:.3f}GB)  → 건너뜀')
else:
    print(f'복사 중: {src.name}  ({src.stat().st_size/1e9:.3f}GB) …')
    shutil.copy(str(src), str(dst))

    # 크기 검증 (manifest 있으면)
    zip_name = CHUNK_NAME.rsplit('.', 1)[0]   # 'train_50k_256.zip'
    if zip_name in _manifest:
        expected = {c['name']: c['bytes'] for c in _manifest[zip_name]['chunks']}
        exp_sz = expected.get(CHUNK_NAME)
        if exp_sz and dst.stat().st_size != exp_sz:
            raise AssertionError(
                f'크기 불일치: {CHUNK_NAME}\n'
                f'  expected {exp_sz/1e9:.3f}GB  got {dst.stat().st_size/1e9:.3f}GB\n'
                f'Drive 업로드가 잘렸습니다 — 다시 업로드하세요.'
            )
    print(f'완료: {dst}  ({dst.stat().st_size/1e9:.3f}GB) ✓')

# 현재 /content/ 에 있는 chunk 목록 출력
import glob
for z in ['train', 'valid']:
    chunks = sorted(glob.glob(f'/content/{z}_*_{DATA_RES}.zip*'))
    if chunks:
        print(f'  {z}: {[Path(c).name for c in chunks]}')

### 3-c. Merge + Extract  (모든 chunk 복사 완료 후 1회 실행)

In [ ]:
### 3-c-1. Merge train chunks → /content/train_50k_<RES>.zip

def _merge(zip_name: str) -> str:
    """Merge /content/<zip_name>.000, .001, … into /content/<zip_name>.
    Returns the merged zip path. Skips if already merged."""
    merged = Path(f'/content/{zip_name}')
    if merged.exists():
        print(f'이미 존재: {merged}  ({merged.stat().st_size/1e9:.2f}GB)  → 건너뜀')
        return str(merged)

    chunks = sorted(glob.glob(f'/content/{zip_name}.[0-9][0-9][0-9]'))
    assert chunks, (
        f'/content/ 에 {zip_name}.000 ~ 이 없습니다.\n'
        f'3-b 셀을 반복 실행해 모든 chunk를 먼저 복사하세요.'
    )

    print(f'Merging {len(chunks)} chunks → {merged} …')
    with open(merged, 'wb') as out:
        for chunk in chunks:
            sz = Path(chunk).stat().st_size
            print(f'  + {Path(chunk).name}  ({sz/1e9:.3f}GB)')
            with open(chunk, 'rb') as src:
                shutil.copyfileobj(src, out, 64 * 1024 * 1024)

    actual = merged.stat().st_size
    if zip_name in _manifest:
        expected = _manifest[zip_name]['total_bytes']
        assert actual == expected, (
            f'크기 불일치: expected {expected/1e9:.3f}GB  got {actual/1e9:.3f}GB\n'
            f'누락되거나 잘린 chunk가 있습니다. 3-b 셀로 다시 확인하세요.'
        )
    print(f'Merge 완료: {actual/1e9:.2f}GB ✓')
    return str(merged)


train_zip_name = f'train_50k_{DATA_RES}.zip'
train_zip_path = _merge(train_zip_name)

In [ ]:
### 3-c-2. Merge valid chunks → /content/valid_10k_<RES>.zip

valid_zip_name = f'valid_10k_{DATA_RES}.zip'
valid_zip_path = _merge(valid_zip_name)

In [ ]:
### 3-c-3. Extract → /content/ffhq_<RES>/train  &  /valid

if not train_dir.exists() or len(list(train_dir.iterdir())) < 100:
    print(f'Extracting train …')
    parallel_unzip(train_zip_path, train_dir, strip_dirs=True, stage_local=True)
else:
    print(f'Train already extracted: {len(list(train_dir.iterdir()))} imgs')

if not valid_dir.exists() or len(list(valid_dir.iterdir())) < 100:
    print(f'Extracting valid …')
    parallel_unzip(valid_zip_path, valid_dir, strip_dirs=True, stage_local=True)
else:
    print(f'Valid already extracted: {len(list(valid_dir.iterdir()))} imgs')

TRAIN_ROOT = str(train_dir)
VALID_ROOT = str(valid_dir)
print(f'TRAIN_ROOT = {TRAIN_ROOT}  ({len(list(train_dir.iterdir()))} imgs)')
print(f'VALID_ROOT = {VALID_ROOT}  ({len(list(valid_dir.iterdir()))} imgs)')

## 4. WandB login

In [ ]:
import wandb

# ── USER CONFIG ──────────────────────────────────────────────────────────────
WANDB_API_KEY = 'YOUR_WANDB_API_KEY_HERE'
WANDB_PROJECT = 'project02-stylegan2'
WANDB_ENTITY  = None   # your WandB username/org, or None
# ────────────────────────────────────────────────────────────────────────────

wandb.login(key=WANDB_API_KEY)
print('WandB logged in.')

import yaml
from types import SimpleNamespace
from src.models.generator import StyleGAN2Generator
from src.models.discriminator import StyleGAN2Discriminator
from src.training.trainer import Trainer, restore_from_drive, find_latest_checkpoint
from src.data.dataset import build_dataloader
from src.utils.fid_score import ValidFIDCache

def load_cfg(yaml_path):
    with open(yaml_path) as f:
        return SimpleNamespace(**yaml.safe_load(f))

CFG_DIR  = f'{REPO_DIR}/configs'
CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
device = torch.device('cuda')

# Sanity-check parameter count
G_check = StyleGAN2Generator(resolution=1024)
print(f'G params: {G_check.count_parameters():,} ({G_check.count_parameters()/1e6:.3f}M)  [limit: 40M]')
assert G_check.count_parameters() < 40_000_000
del G_check

In [ ]:
import yaml
from types import SimpleNamespace
from src.models.generator import StyleGAN2Generator
from src.models.discriminator import StyleGAN2Discriminator
from src.training.trainer import Trainer
from src.data.dataset import build_dataloader
from src.utils.fid_score import ValidFIDCache

def load_cfg(yaml_path):
    with open(yaml_path) as f:
        return SimpleNamespace(**yaml.safe_load(f))

CFG_DIR  = f'{REPO_DIR}/configs'
CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
device = torch.device('cuda')

# Sanity-check parameter count
G_check = StyleGAN2Generator(resolution=1024)
print(f'G params: {G_check.count_parameters():,} ({G_check.count_parameters()/1e6:.3f}M)  [limit: 40M]')
assert G_check.count_parameters() < 40_000_000
del G_check

cfg256 = load_cfg(f'{CFG_DIR}/train_256.yaml')

train_loader = build_dataloader(TRAIN_ROOT, None, cfg256.resolution, cfg256.batch_size, cfg256.num_workers, aug=True)
valid_loader = build_dataloader(VALID_ROOT, None, cfg256.resolution, cfg256.batch_size, cfg256.num_workers, aug=False)

run256 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                    name='train-256', config=vars(cfg256), resume='allow')

trainer256 = Trainer(cfg256)
restore_from_drive(trainer256, DRIVE_CKPT_DIR, CKPT_DIR, resolution=256)

trainer256.fit(train_loader, valid_loader,
               wandb_run=run256,
               drive_backup_dir=DRIVE_CKPT_DIR,
               ckpt_dir=CKPT_DIR)
run256.finish()
CKPT_256 = f'{CKPT_DIR}/ckpt_256_final.pth'
print('Stage 1 done →', CKPT_256)

In [ ]:
cfg256 = load_cfg(f'{CFG_DIR}/train_256.yaml')

train_loader = build_dataloader(TRAIN_ROOT, None, cfg256.resolution, cfg256.batch_size, cfg256.num_workers, aug=True)
valid_loader = build_dataloader(VALID_ROOT, None, cfg256.resolution, cfg256.batch_size, cfg256.num_workers, aug=False)

run256 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                    name='train-256', config=vars(cfg256), resume='allow')

trainer256 = Trainer(cfg256)

# ── Auto-resume from Drive checkpoint ───────────────────────────────────────
# Detects the latest backup in Drive and restores model + optimizer + step.
# If no checkpoint exists, training starts from scratch automatically.
from src.training.trainer import restore_from_drive
restore_from_drive(trainer256, DRIVE_CKPT_DIR, CKPT_DIR, resolution=256)
# ────────────────────────────────────────────────────────────────────────────

trainer256.fit(train_loader, valid_loader,
               wandb_run=run256,
               drive_backup_dir=DRIVE_CKPT_DIR,
               ckpt_dir=CKPT_DIR)
run256.finish()
CKPT_256 = f'{CKPT_DIR}/ckpt_256_final.pth'
print('Stage 1 done →', CKPT_256)

cfg512 = load_cfg(f'{CFG_DIR}/train_512.yaml')

train_loader_512 = build_dataloader(TRAIN_ROOT, None, cfg512.resolution, cfg512.batch_size, cfg512.num_workers, aug=True)
valid_loader_512 = build_dataloader(VALID_ROOT, None, cfg512.resolution, cfg512.batch_size, cfg512.num_workers, aug=False)

run512 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                    name='train-512', config=vars(cfg512), resume='allow')

trainer512 = Trainer(cfg512)
resumed = restore_from_drive(trainer512, DRIVE_CKPT_DIR, CKPT_DIR, resolution=512)

if not resumed:
    ckpt_256_drive = find_latest_checkpoint(DRIVE_CKPT_DIR, resolution=256)
    assert ckpt_256_drive is not None, f'No 256 checkpoint in {DRIVE_CKPT_DIR}'
    ckpt_256_local = f'{CKPT_DIR}/ckpt_256_for_512.pth'
    print(f'Loading 256 checkpoint: {ckpt_256_drive}')
    shutil.copy(ckpt_256_drive, ckpt_256_local)
    state = torch.load(ckpt_256_local, map_location=device)
    trainer512.G.load_from_lower_resolution(state['G'])
    trainer512.D.load_from_lower_resolution(state['D'])

trainer512.fit(train_loader_512, valid_loader_512,
               wandb_run=run512,
               drive_backup_dir=DRIVE_CKPT_DIR,
               ckpt_dir=CKPT_DIR)
run512.finish()
CKPT_512 = f'{CKPT_DIR}/ckpt_512_final.pth'
print('Stage 2 done →', CKPT_512)

In [ ]:
cfg512 = load_cfg(f'{CFG_DIR}/train_512.yaml')

train_loader_512 = build_dataloader(TRAIN_ROOT, None, cfg512.resolution, cfg512.batch_size, cfg512.num_workers, aug=True)
valid_loader_512 = build_dataloader(VALID_ROOT, None, cfg512.resolution, cfg512.batch_size, cfg512.num_workers, aug=False)

run512 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                    name='train-512', config=vars(cfg512), resume='allow')

trainer512 = Trainer(cfg512)

# ── Auto-resume: try 512 Drive checkpoint first ──────────────────────────────
resumed = restore_from_drive(trainer512, DRIVE_CKPT_DIR, CKPT_DIR, resolution=512)

if not resumed:
    # No 512 checkpoint found → first run of Stage 2.
    # Find the latest 256 checkpoint in Drive automatically
    # (handles both ckpt_256_final.pth and ckpt_256_0080000.pth style names).
    from src.training.trainer import find_latest_checkpoint
    ckpt_256_drive = find_latest_checkpoint(DRIVE_CKPT_DIR, resolution=256)
    assert ckpt_256_drive is not None, (
        f'No 256 checkpoint found in {DRIVE_CKPT_DIR}. '
        'Make sure Stage 1 training is complete and the checkpoint is backed up.'
    )
    ckpt_256_local = f'{CKPT_DIR}/ckpt_256_for_512.pth'
    print(f'Loading 256 checkpoint: {ckpt_256_drive}')
    shutil.copy(ckpt_256_drive, ckpt_256_local)
    state = torch.load(ckpt_256_local, map_location=device)
    trainer512.G.load_from_lower_resolution(state['G'])
    trainer512.D.load_state_dict(state['D'], strict=False)
    print('256 weights loaded into 512 model (new 512/1024 blocks are random-init).')
# ────────────────────────────────────────────────────────────────────────────

trainer512.fit(train_loader_512, valid_loader_512,
               wandb_run=run512,
               drive_backup_dir=DRIVE_CKPT_DIR,
               ckpt_dir=CKPT_DIR)
run512.finish()
CKPT_512 = f'{CKPT_DIR}/ckpt_512_final.pth'
print('Stage 2 done →', CKPT_512)

cfg1024 = load_cfg(f'{CFG_DIR}/train_1024.yaml')

train_loader_1024 = build_dataloader(TRAIN_ROOT, None, cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers, aug=True)
valid_loader_1024 = build_dataloader(VALID_ROOT, None, cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers, aug=False)

run1024 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name='train-1024', config=vars(cfg1024), resume='allow')

trainer1024 = Trainer(cfg1024)
resumed = restore_from_drive(trainer1024, DRIVE_CKPT_DIR, CKPT_DIR, resolution=1024)

if not resumed:
    ckpt_512_drive = find_latest_checkpoint(DRIVE_CKPT_DIR, resolution=512)
    assert ckpt_512_drive is not None, f'No 512 checkpoint in {DRIVE_CKPT_DIR}'
    ckpt_512_local = f'{CKPT_DIR}/ckpt_512_for_1024.pth'
    print(f'Loading 512 checkpoint: {ckpt_512_drive}')
    shutil.copy(ckpt_512_drive, ckpt_512_local)
    state = torch.load(ckpt_512_local, map_location=device)
    trainer1024.G.load_from_lower_resolution(state['G'])
    trainer1024.D.load_from_lower_resolution(state['D'])

trainer1024.fit(train_loader_1024, valid_loader_1024,
                wandb_run=run1024,
                drive_backup_dir=DRIVE_CKPT_DIR,
                ckpt_dir=CKPT_DIR)
run1024.finish()
CKPT_1024 = f'{CKPT_DIR}/ckpt_1024_final.pth'
print('Stage 3 done →', CKPT_1024)

In [ ]:
cfg1024 = load_cfg(f'{CFG_DIR}/train_1024.yaml')

train_loader_1024 = build_dataloader(TRAIN_ROOT, None, cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers, aug=True)
valid_loader_1024 = build_dataloader(VALID_ROOT, None, cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers, aug=False)

run1024 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name='train-1024', config=vars(cfg1024), resume='allow')

trainer1024 = Trainer(cfg1024)

# ── Auto-resume: try 1024 Drive checkpoint first ─────────────────────────────
resumed = restore_from_drive(trainer1024, DRIVE_CKPT_DIR, CKPT_DIR, resolution=1024)

if not resumed:
    # No 1024 checkpoint found → first run of Stage 3.
    # Find the latest 512 checkpoint in Drive automatically.
    ckpt_512_drive = find_latest_checkpoint(DRIVE_CKPT_DIR, resolution=512)
    assert ckpt_512_drive is not None, (
        f'No 512 checkpoint found in {DRIVE_CKPT_DIR}. '
        'Make sure Stage 2 training is complete and the checkpoint is backed up.'
    )
    ckpt_512_local = f'{CKPT_DIR}/ckpt_512_for_1024.pth'
    print(f'Loading 512 checkpoint: {ckpt_512_drive}')
    shutil.copy(ckpt_512_drive, ckpt_512_local)
    state = torch.load(ckpt_512_local, map_location=device)
    trainer1024.G.load_from_lower_resolution(state['G'])
    trainer1024.D.load_state_dict(state['D'], strict=False)
    print('512 weights loaded into 1024 model (new 1024 block is random-init).')
# ────────────────────────────────────────────────────────────────────────────

trainer1024.fit(train_loader_1024, valid_loader_1024,
                wandb_run=run1024,
                drive_backup_dir=DRIVE_CKPT_DIR,
                ckpt_dir=CKPT_DIR)
run1024.finish()
CKPT_1024 = f'{CKPT_DIR}/ckpt_1024_final.pth'
print('Stage 3 done →', CKPT_1024)

## 9. FID evaluation (valid set)

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
EVAL_CKPT = CKPT_1024
EVAL_RES  = 1024
# ────────────────────────────────────────────────────────────────────────────

cfg_eval = load_cfg(f'{CFG_DIR}/train_{EVAL_RES}.yaml')
G_eval = StyleGAN2Generator(
    resolution=cfg_eval.resolution, z_dim=cfg_eval.z_dim,
    w_dim=cfg_eval.w_dim, channel_base=cfg_eval.channel_base,
    channel_max=cfg_eval.channel_max, mapping_layers=cfg_eval.mapping_layers,
).to(device)
G_eval.load_state_dict(torch.load(EVAL_CKPT, map_location=device)['G'])
G_eval.eval()
print(f'G params: {G_eval.count_parameters():,}')

valid_loader_eval = build_dataloader(VALID_ROOT, None, EVAL_RES, 8, 4, aug=False)
fid_cache = ValidFIDCache(valid_loader_eval, device)
fid = fid_cache.compute(G_eval, n_gen=10000, batch_size=16)
print(f'FID (valid, 10k): {fid:.2f}')

## 10. Generate samples

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

mean_w = G_eval.mapping.mean_latent(n_samples=4096, device=str(device))
with torch.no_grad():
    z = torch.randn(16, cfg_eval.z_dim, device=device)
    imgs = G_eval(z, noise_mode='const', truncation=0.7, mean_w=mean_w)
    imgs = (imgs.clamp(-1, 1) + 1) / 2

grid = make_grid(imgs.cpu(), nrow=4, padding=2)
plt.figure(figsize=(14, 14))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title(f'StyleGAN2 {EVAL_RES}×{EVAL_RES}  truncation=0.7')
plt.tight_layout()
plt.savefig('/content/samples.png', dpi=150)
plt.show()

## 11. Export ONNX (submission)

In [ ]:
import shutil
ONNX_PATH = f'/content/generator_{EVAL_RES}.onnx'
onnx_params = G_eval.export_onnx(ONNX_PATH, batch_size=1)
shutil.copy(ONNX_PATH, f'{DRIVE_DIR}/generator_{EVAL_RES}.onnx')
print(f'Saved to Drive: {DRIVE_DIR}/generator_{EVAL_RES}.onnx')